# DAY 183 — GitHub Actions CI/CD for Data Pipelines (Part 1)
### Month 11: Data Engineering & ETL Pipelines | LogiTrack India Dataset (seed=181)

**Prerequisite check:** Days 181–182 (raw data generation, ETL transformation, scheduling, error handling) — this day ONLY uses those skills + new CI/CD concepts. No new Python syntax beyond what's introduced below.

**Today's scope:** Writing a standalone data-quality validation script, understanding what GitHub Actions does for a data pipeline, writing your first workflow YAML, and simulating a CI run locally (since GitHub Actions itself runs on GitHub's servers, not in this notebook).

## SECTION 1: Raw Data (DO NOT MODIFY)
This is the same LogiTrack India generation logic used on Day 181/182. Treat this cell as frozen — never edit it directly in any task below.

In [1]:
# ============================================
# RAW DATA — DO NOT MODIFY THIS CELL
# LogiTrack India — Logistics/Shipment dataset
# ============================================
import pandas as pd
import numpy as np

np.random.seed(181)

n_rows = 530

cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kolkata", "Ahmedabad"]
carriers = ["BlueDart", "Delhivery", "Ekart", "DTDC", "XpressBees"]
status_options = ["Delivered", "In Transit", "Delayed", "Returned", "Pending"]

raw_df = pd.DataFrame({
    "shipment_id": [f"SHIP{1000+i}" for i in range(n_rows)],
    "origin_city": np.random.choice(cities, n_rows),
    "destination_city": np.random.choice(cities, n_rows),
    "carrier": np.random.choice(carriers, n_rows),
    "status": np.random.choice(status_options, n_rows, p=[0.55, 0.20, 0.12, 0.06, 0.07]),
    "weight_kg": np.round(np.random.uniform(0.5, 80, n_rows), 2),
    "shipping_cost_inr": np.round(np.random.uniform(50, 3500, n_rows), 2),
    "transit_days": np.random.randint(1, 12, n_rows),
    "ship_date": pd.date_range("2025-01-01", periods=n_rows, freq="9h").astype(str)
})

# inject some raw-world messiness (consistent with Day 181 raw cell)
raw_df.loc[np.random.choice(n_rows, 18, replace=False), "weight_kg"] = np.nan
raw_df.loc[np.random.choice(n_rows, 10, replace=False), "shipping_cost_inr"] = np.nan
dupe_idx = np.random.choice(n_rows, 6, replace=False)
raw_df = pd.concat([raw_df, raw_df.loc[dupe_idx]], ignore_index=True)

print(f"Raw shape: {raw_df.shape}")
raw_df.head()

Raw shape: (536, 9)


,shipment_id,origin_city,destination_city,carrier,status,weight_kg,shipping_cost_inr,transit_days,ship_date
0,SHIP1000,Pune,Pune,XpressBees,Delivered,7.93,1966.38,4,2025-01-01 00:00:00
1,SHIP1001,Delhi,Pune,DTDC,Pending,13.04,1823.20,8,2025-01-01 09:00:00
2,SHIP1002,Delhi,Bangalore,DTDC,Delivered,49.41,3272.00,4,2025-01-01 18:00:00
3,SHIP1003,Delhi,Bangalore,XpressBees,Returned,42.00,1815.87,3,2025-01-02 03:00:00
4,SHIP1004,Kolkata,Ahmedabad,DTDC,Delivered,63.55,3315.08,2,2025-01-02 12:00:00


## SECTION 2: Concept Notes

### What is CI/CD, and why does a data pipeline need it?
**CI (Continuous Integration)** means every time you push code (e.g. your ETL script), an automated process runs checks on it BEFORE it's trusted — without you manually running it yourself.

**CD (Continuous Deployment/Delivery)** means once those checks pass, the validated code/pipeline can automatically move toward production (e.g. trigger the next pipeline run, deploy a dashboard refresh).

### Why a freelance data engineer cares about this
A client paying for a recurring pipeline (e.g. daily LogiTrack shipment report) does not want to trust "it worked on my laptop." They want proof that:
1. The script ran without error
2. The output data meets quality rules (no nulls in critical columns, expected row count, correct schema)
3. This check happens automatically on every code change — not only when you remember to test

This is the difference between a freelancer who hands over a script, and one who hands over a **pipeline with a safety net**. Clients pay premium rates for the safety net.

### GitHub Actions — the mechanism
GitHub Actions is GitHub's built-in automation runner. You define a **workflow** in a YAML file at `.github/workflows/*.yml`. GitHub triggers it on events (push, pull request, schedule) and runs your commands on a temporary virtual machine.

Key vocabulary:
| Term | Meaning |
|---|---|
| Workflow | The whole automated process (one YAML file) |
| Trigger (`on:`) | What causes the workflow to run (push, schedule, manual) |
| Job | A group of steps that run on one virtual machine |
| Step | One command or action (e.g. install deps, run script) |
| Runner | The virtual machine executing the job (e.g. `ubuntu-latest`) |

### Today's two deliverables
1. **`data_quality_check.py`** — a standalone, importable validation script (this is what CI will run)
2. **`.github/workflows/data_quality.yml`** — the workflow definition that tells GitHub to run that script automatically

Since this notebook environment can't talk to GitHub's servers, you will **simulate the CI run locally** by executing the script exactly as the workflow would — this proves the logic is correct before you ever push it.

## SECTION 3: Practice Guide — Tasks

**Rules:** Use only `raw_df` from Section 1 (read-only). Write all logic in the cells below. Do not import anything beyond pandas, numpy, os, sys, subprocess.

---
### Task 1 (15 pts) — Write the ETL step the CI will validate
Recreate the cleaning steps from Day 181/182 (drop duplicates, fill `weight_kg` median, fill `shipping_cost_inr` median) into a function `clean_logitrack(df)` that returns a cleaned DataFrame. This represents "the pipeline output" CI needs to check.

### Task 2 (25 pts) — Build the data quality validation function
Write a function `validate_data_quality(df)` that returns a dict with these checks (each True/False):
- `"no_nulls_critical"`: True if `weight_kg`, `shipping_cost_inr`, `shipment_id` have ZERO nulls
- `"row_count_ok"`: True if row count is between 500 and 530 (allowing for legitimate dedup loss, but catching catastrophic data loss)
- `"no_duplicate_ids"`: True if `shipment_id` has zero duplicates
- `"schema_ok"`: True if all 9 expected columns are present (list them explicitly)
- `"cost_positive"`: True if `shipping_cost_inr` has no values <= 0

The function must also return an overall `"all_passed"` key (True only if every check above is True).

### Task 3 (15 pts) — Write `data_quality_check.py` to disk
Using Python's file-writing (`with open(...) as f: f.write(...)`), save a STANDALONE script (not dependent on notebook variables) to `data_quality_check.py` that:
- Regenerates/loads the data (you can re-use the raw generation logic inline)
- Runs your clean + validate functions
- Prints each check result
- Exits with `sys.exit(1)` if `all_passed` is False, `sys.exit(0)` if True

This is the exact script GitHub Actions will execute — it must run independently with `python data_quality_check.py`.

### Task 4 (15 pts) — Simulate the CI run locally
Use the `subprocess` module to run `python data_quality_check.py` from within the notebook and capture/print the output and exit code. This proves what GitHub Actions would see.

### Task 5 (20 pts) — Write the GitHub Actions workflow YAML
Write (as a string, saved to `data_quality_workflow.yml`) a valid workflow that:
- Is named `Data Quality Check`
- Triggers `on: push` to branch `master` AND `on: workflow_dispatch` (manual trigger)
- Has one job `validate` running on `ubuntu-latest`
- Steps: checkout code (`actions/checkout@v4`), set up Python 3.11 (`actions/setup-python@v5`), install pandas/numpy, run `python data_quality_check.py`

### Task 6 (10 pts) — NRA Business Summary
Using the actual validation results your script printed (read directly from output, not estimated), write ONE NRA bullet:
**Number** = exact pass/fail counts and which checks failed (if any), pulled from printed output only.
**Reason** = the causal mechanism for the failure/success state (e.g. why nulls remained, or why the run was clean) — not a description of the outcome.
**Action** = the specific, committed next step (e.g. exact fix to add to the cleaning function, or "no action — pipeline cleared for production deployment").

---
### Comment skeletons to start from:
```python
# Task 1
def clean_logitrack(df):
    # your code here
    pass

# Task 2
def validate_data_quality(df):
    checks = {}
    # your code here
    checks["all_passed"] = all(checks.values())
    return checks

# Task 3 — write the standalone script
script_content = '''
# full standalone script text here
'''
with open("data_quality_check.py", "w") as f:
    f.write(script_content)

# Task 4
import subprocess
result = subprocess.run(["python", "data_quality_check.py"], capture_output=True, text=True)
print(result.stdout)
print("Exit code:", result.returncode)

# Task 5
yaml_content = '''
name: Data Quality Check
on:
  push:
    branches: [master]
  workflow_dispatch:
jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pandas numpy
      - run: python data_quality_check.py
'''
with open("data_quality_workflow.yml", "w") as f:
    f.write(yaml_content)
```

## Your Work Starts Below

In [2]:
# =============================================================================
# TASK 1: clean_logitrack(df) – ETL cleaning step (15 pts)
# Goal: Deduplicate on shipment_id, fill nulls in weight_kg and shipping_cost_inr with median.
# Method: Drop duplicates keeping first occurrence, then fill NaNs with column median.
# =============================================================================

def clean_logitrack(df):
    """
    Clean the raw LogiTrack dataframe:
      - Deduplicate on shipment_id (keep first)
      - Fill missing weight_kg with median
      - Fill missing shipping_cost_inr with median
    Returns cleaned DataFrame.
    """
    # 1. Deduplicate on shipment_id (assume first occurrence is the correct one)
    df_clean = df.drop_duplicates(subset='shipment_id', keep='first').copy()
    
    # 2. Fill missing weight_kg with median (ignore NaNs in median calculation)
    median_weight = df_clean['weight_kg'].median()
    df_clean['weight_kg'] = df_clean['weight_kg'].fillna(median_weight)
    
    # 3. Fill missing shipping_cost_inr with median
    median_cost = df_clean['shipping_cost_inr'].median()
    df_clean['shipping_cost_inr'] = df_clean['shipping_cost_inr'].fillna(median_cost)
    
    return df_clean

In [3]:
# =============================================================================
# TASK 2: validate_data_quality(df) – build quality checks (25 pts)
# Goal: Return dict of True/False for five checks, plus overall all_passed.
# Method: Check each condition explicitly; all_passed = all(checks.values()).
# =============================================================================

def validate_data_quality(df):
    """
    Validate the cleaned DataFrame against defined quality rules.
    Returns a dictionary with each check result and an overall 'all_passed' key.
    """
    checks = {}
    
    # Expected columns (9 total)
    expected_cols = [
        'shipment_id', 'origin_city', 'destination_city', 'carrier', 
        'status', 'weight_kg', 'shipping_cost_inr', 'transit_days', 'ship_date'
    ]
    
    # 1. No nulls in critical columns: shipment_id, weight_kg, shipping_cost_inr
    checks['no_nulls_critical'] = (
        df['shipment_id'].isnull().sum() == 0 and
        df['weight_kg'].isnull().sum() == 0 and
        df['shipping_cost_inr'].isnull().sum() == 0
    )
    
    # 2. Row count between 500 and 530 (allowing for legitimate dedup loss)
    row_count = len(df)
    checks['row_count_ok'] = (500 <= row_count <= 530)
    
    # 3. No duplicate shipment_id
    checks['no_duplicate_ids'] = df['shipment_id'].duplicated().sum() == 0
    
    # 4. Schema: all 9 expected columns present
    checks['schema_ok'] = all(col in df.columns for col in expected_cols)
    
    # 5. All shipping costs are positive (>0)
    checks['cost_positive'] = (df['shipping_cost_inr'] > 0).all()
    
    # Overall pass/fail
    checks['all_passed'] = all(checks.values())
    
    return checks

In [7]:
# =============================================================================
# TASK 3: Write standalone script to disk (15 pts)
# Goal: Save a self‑contained Python script that regenerates data, cleans, validates,
#       prints results, and exits with appropriate sys.exit code.
# Method: Use raw generation logic identical to notebook cell; include clean & validate.
# =============================================================================

script_content = '''#!/usr/bin/env python3
# data_quality_check.py - Standalone CI validation script   # fixed: en dash -> hyphen
# Purpose: Regenerate LogiTrack data, clean it, run quality checks, and exit with code 0/1.

import pandas as pd
import numpy as np
import sys

# -------------------- RAW DATA GENERATION (same as notebook) --------------------
np.random.seed(181)
n_rows = 530

cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Hyderabad", "Pune", "Kolkata", "Ahmedabad"]
carriers = ["BlueDart", "Delhivery", "Ekart", "DTDC", "XpressBees"]
status_options = ["Delivered", "In Transit", "Delayed", "Returned", "Pending"]

raw_df = pd.DataFrame({
    "shipment_id": [f"SHIP{1000+i}" for i in range(n_rows)],
    "origin_city": np.random.choice(cities, n_rows),
    "destination_city": np.random.choice(cities, n_rows),
    "carrier": np.random.choice(carriers, n_rows),
    "status": np.random.choice(status_options, n_rows, p=[0.55, 0.20, 0.12, 0.06, 0.07]),
    "weight_kg": np.round(np.random.uniform(0.5, 80, n_rows), 2),
    "shipping_cost_inr": np.round(np.random.uniform(50, 3500, n_rows), 2),
    "transit_days": np.random.randint(1, 12, n_rows),
    "ship_date": pd.date_range("2025-01-01", periods=n_rows, freq="9h").astype(str)
})

# inject some messiness (consistent with notebook)
raw_df.loc[np.random.choice(n_rows, 18, replace=False), "weight_kg"] = np.nan
raw_df.loc[np.random.choice(n_rows, 10, replace=False), "shipping_cost_inr"] = np.nan
dupe_idx = np.random.choice(n_rows, 6, replace=False)
raw_df = pd.concat([raw_df, raw_df.loc[dupe_idx]], ignore_index=True)

# -------------------- CLEAN FUNCTION (same as Task 1) --------------------
def clean_logitrack(df):
    df_clean = df.drop_duplicates(subset='shipment_id', keep='first').copy()
    median_weight = df_clean['weight_kg'].median()
    df_clean['weight_kg'] = df_clean['weight_kg'].fillna(median_weight)
    median_cost = df_clean['shipping_cost_inr'].median()
    df_clean['shipping_cost_inr'] = df_clean['shipping_cost_inr'].fillna(median_cost)
    return df_clean

# -------------------- VALIDATE FUNCTION (same as Task 2) --------------------
def validate_data_quality(df):
    checks = {}
    expected_cols = [
        'shipment_id', 'origin_city', 'destination_city', 'carrier',
        'status', 'weight_kg', 'shipping_cost_inr', 'transit_days', 'ship_date'
    ]
    checks['no_nulls_critical'] = (
        df['shipment_id'].isnull().sum() == 0 and
        df['weight_kg'].isnull().sum() == 0 and
        df['shipping_cost_inr'].isnull().sum() == 0
    )
    row_count = len(df)
    checks['row_count_ok'] = (500 <= row_count <= 530)
    checks['no_duplicate_ids'] = df['shipment_id'].duplicated().sum() == 0
    checks['schema_ok'] = all(col in df.columns for col in expected_cols)
    checks['cost_positive'] = (df['shipping_cost_inr'] > 0).all()
    checks['all_passed'] = all(checks.values())
    return checks

# -------------------- MAIN EXECUTION --------------------
if __name__ == "__main__":
    print("Running data quality checks...")
    
    # Clean the raw data
    clean_df = clean_logitrack(raw_df)
    print(f"Cleaned row count: {len(clean_df)}")
    
    # Validate
    results = validate_data_quality(clean_df)
    
    # Print each check result
    for check_name, passed in results.items():
        print(f"{check_name}: {passed}")
    
    # Exit with appropriate code
    if results['all_passed']:
        print("All quality checks passed. Exiting with code 0.")
        sys.exit(0)
    else:
        print("Quality checks failed. Exiting with code 1.")
        sys.exit(1)
'''

# Write the script to disk with explicit UTF-8 encoding
with open("data_quality_check.py", "w", encoding='utf-8') as f:
    f.write(script_content)
    
print("data_quality_check.py written successfully.")

data_quality_check.py written successfully.


In [8]:
# =============================================================================
# TASK 4: Simulate CI run via subprocess (15 pts)
# Goal: Run the standalone script and capture stdout/exit code.
# Method: Use subprocess.run with capture_output=True, print results.
# =============================================================================

import subprocess

print("Simulating GitHub Actions run by executing data_quality_check.py...")
print("-" * 50)

result = subprocess.run(
    ["python", "data_quality_check.py"],
    capture_output=True,
    text=True
)

# Print stdout (which includes all print statements from the script)
print(result.stdout)

# Print stderr if any (should be empty)
if result.stderr:
    print("stderr:")
    print(result.stderr)

print(f"Exit code: {result.returncode}")
print("-" * 50)

# Interpret the exit code for human readability
if result.returncode == 0:
    print("✅ Local CI simulation PASSED (exit code 0)")
else:
    print("❌ Local CI simulation FAILED (exit code 1)")

Simulating GitHub Actions run by executing data_quality_check.py...
--------------------------------------------------
Running data quality checks...
Cleaned row count: 530
no_nulls_critical: True
row_count_ok: True
no_duplicate_ids: True
schema_ok: True
cost_positive: True
all_passed: True
All quality checks passed. Exiting with code 0.

Exit code: 0
--------------------------------------------------
✅ Local CI simulation PASSED (exit code 0)


In [9]:
# =============================================================================
# TASK 5: Write workflow YAML to disk (20 pts)
# Goal: Create a valid GitHub Actions workflow file as a string and save it.
# Method: Write YAML content with correct triggers, job, steps.
# =============================================================================

yaml_content = '''
name: Data Quality Check

on:
  push:
    branches: [ master ]
  workflow_dispatch:   # allow manual trigger from GitHub UI

jobs:
  validate:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          pip install pandas numpy

      - name: Run data quality check
        run: python data_quality_check.py
'''

# Save the workflow file – note: in a real repo it would be .github/workflows/data_quality.yml
# Here we save it in the current directory for demonstration.
with open("data_quality_workflow.yml", "w") as f:
    f.write(yaml_content)

print("data_quality_workflow.yml written successfully.")
print("(In production, place this file in .github/workflows/ directory)")

data_quality_workflow.yml written successfully.
(In production, place this file in .github/workflows/ directory)


**Task 6 — NRA Business Summary:**

**Bullet 1 — [Data Quality Validation Passed]**

**Number** : 530 rows cleaned, all 5 quality checks passed: `no_nulls_critical: True`, `row_count_ok: True`, `no_duplicate_ids: True`, `schema_ok: True`, `cost_positive: True` (summary flag `all_passed: True`).

**Reason** : The cleaning pipeline deduplicates on `shipment_id` and replaces missing `weight_kg` and `shipping_cost_inr` with the column median, which eliminates all nulls in critical columns, reduces row count from 536 to exactly 530 (dropping 6 duplicate IDs), and ensures all shipping costs remain positive; the validation passes because these transformations correctly handle the injected data messiness (missing values and duplicate rows) introduced during generation.

**Action** : Deploy the current ETL and validation scripts to production; add the `data_quality_check.py` script as a mandatory step in the GitHub Actions workflow triggered on every push to the `master` branch, and configure the workflow to block any merge if this check fails.

## SECTION 4: Scoring Rubric

| Task | Points | Criteria |
|---|---|---|
| Task 1 — clean_logitrack() | 15 | Correct dedup on shipment_id, correct median fill on both columns |
| Task 2 — validate_data_quality() | 25 | All 5 checks correctly logic'd + all_passed key correctly computed |
| Task 3 — standalone script written | 15 | Script is self-contained (no notebook variable dependency), correct sys.exit logic |
| Task 4 — subprocess simulation | 15 | Correctly captures stdout + returncode, output printed |
| Task 5 — workflow YAML | 20 | Correct triggers, job name, runner, all 4 steps present and ordered correctly |
| Task 6 — NRA summary | 10 | Number from printed output only; Reason = causal mechanism; Action = specific & committed |
| **Total** | **100** | +10★ bonus for zero deductions across all NRA/technical checks |

**Common deduction triggers (per project NRA discipline):**
- Reason describing the outcome ("the checks passed") instead of the mechanism (WHY they passed) → -3
- Action with hedging language ("should probably", "could consider") → -3
- Script depends on notebook-scope variables instead of being truly standalone → -5 (defeats the purpose of CI — GitHub's runner has no access to your notebook)
- Workflow file placed in wrong path or wrong branch name → flagged per project convention (workflow must be `.github/workflows/`, repo root, branch `master`)

## Interview Framing

**Q: "Walk me through how you'd add automated testing to a data pipeline you're delivering to a client."**

Model answer structure: *"I separate the pipeline logic from the validation logic — the ETL script transforms the data, and a standalone data-quality script checks the output against explicit rules: schema, nulls in critical fields, row count bounds, and duplicate keys. I wire that validation script into a GitHub Actions workflow that triggers on every push to the main branch, so the client gets proof the pipeline is healthy without needing to trust a manual test run. The key design choice is making the validation script fully standalone — no dependency on a notebook or local environment state — because the CI runner starts from a clean virtual machine every time."*

This answer signals: separation of concerns, reproducibility awareness, and understanding of *why* CI runners need self-contained scripts — a common signal of production-readiness that distinguishes mid-level from senior freelance work.